# Bulk RNA-seq TME Deconvolution Template

用于 bulk RNA-seq 表达矩阵的免疫/基质评分、细胞比例估计、组间比较和 DEG/GSVA 关联分析。建议输入 normalized expression，例如 TPM、log2(TPM+1)、或 VST 矩阵；CIBERSORT 类工具通常需要非 log TPM。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./1-DEG/vsd_matrix.csv"     # genes x samples; first column is gene name or row names
META_FILE <- "./sample_metadata.csv"       # must contain sample and group columns
EXPR_IS_LOG <- TRUE                         # TRUE for log2(TPM+1)/VST; FALSE for TPM-like input
GENE_COLUMN <- NULL                         # NULL = use rownames/first column; or e.g. "gene"
SAMPLE_COLUMN <- "sample"
GROUP_COLUMN <- "group"
GROUP_LEVELS <- c("Control", "Treatment")

RUN_ESTIMATE <- TRUE
RUN_IOBR <- FALSE                           # IOBR is convenient but has heavier dependencies
RUN_CIBERSORT <- FALSE                      # Requires CIBERSORT.R and LM22/signature file
CIBERSORT_SCRIPT <- "./CIBERSORT.R"
CIBERSORT_SIGNATURE <- "./LM22.txt"

TARGET_SIGNATURES <- c("ImmuneScore", "StromalScore", "ESTIMATEScore")
OUTDIR <- "RNAseq_TME_Deconvolution_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "pheatmap", "ggpubr", "corrplot"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("GSVA", "limma"))
# install.packages("estimate", repos = "http://r-forge.r-project.org")

suppressPackageStartupMessages({
  library(tidyverse)
  library(pheatmap)
  library(ggpubr)
  library(corrplot)
  library(GSVA)
  library(limma)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Expression and Metadata

In [ ]:
expr_raw <- read.csv(EXPR_FILE, check.names = FALSE)
if (!is.null(GENE_COLUMN) && GENE_COLUMN %in% colnames(expr_raw)) {
  genes <- expr_raw[[GENE_COLUMN]]
  expr <- as.matrix(expr_raw[, setdiff(colnames(expr_raw), GENE_COLUMN), drop = FALSE])
  rownames(expr) <- genes
} else if (!is.numeric(expr_raw[[1]])) {
  genes <- expr_raw[[1]]
  expr <- as.matrix(expr_raw[, -1, drop = FALSE])
  rownames(expr) <- genes
} else {
  expr <- as.matrix(expr_raw)
}
mode(expr) <- "numeric"
expr <- expr[!duplicated(rownames(expr)) & !is.na(rownames(expr)) & rownames(expr) != "", , drop = FALSE]

meta <- read.csv(META_FILE, check.names = FALSE)
meta[[GROUP_COLUMN]] <- factor(meta[[GROUP_COLUMN]], levels = GROUP_LEVELS)
common_samples <- intersect(colnames(expr), meta[[SAMPLE_COLUMN]])
expr <- expr[, common_samples, drop = FALSE]
meta <- meta[match(common_samples, meta[[SAMPLE_COLUMN]]), ]
stopifnot(all(colnames(expr) == meta[[SAMPLE_COLUMN]]))
cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")
print(table(meta[[GROUP_COLUMN]], useNA = "ifany"))


## 4. ESTIMATE Score

In [ ]:
if (RUN_ESTIMATE) {
  if (!requireNamespace("estimate", quietly = TRUE)) {
    message("Package 'estimate' is not installed; skipping ESTIMATE.")
  } else {
    estimate_input <- if (EXPR_IS_LOG) 2^expr - 1 else expr
    estimate_df <- data.frame(NAME = rownames(estimate_input), Description = NA, estimate_input, check.names = FALSE)
    write.table(estimate_df, file.path(OUTDIR, "estimate_input.gct"), sep = "\t", quote = FALSE, row.names = FALSE)
    estimate::filterCommonGenes(input.f = file.path(OUTDIR, "estimate_input.gct"),
                                output.f = file.path(OUTDIR, "estimate_common_genes.gct"), id = "GeneSymbol")
    estimate::estimateScore(file.path(OUTDIR, "estimate_common_genes.gct"),
                            file.path(OUTDIR, "estimate_scores.gct"), platform = "illumina")
    estimate_scores <- read.table(file.path(OUTDIR, "estimate_scores.gct"), skip = 2, header = TRUE, sep = "\t", check.names = FALSE)
    rownames(estimate_scores) <- estimate_scores$NAME
    estimate_scores <- as.data.frame(t(estimate_scores[, -(1:2)]))
    estimate_scores[[SAMPLE_COLUMN]] <- rownames(estimate_scores)
    write.csv(estimate_scores, file.path(OUTDIR, "ESTIMATE_scores.csv"), row.names = FALSE)
  }
}


## 5. ssGSEA Immune Signature Scoring

In [ ]:
immune_gene_sets <- list(
  T_cell = c("CD3D", "CD3E", "CD2", "TRAC"),
  CD8_T_cell = c("CD8A", "CD8B", "GZMB", "PRF1"),
  NK_cell = c("NKG7", "GNLY", "KLRD1", "KLRK1"),
  B_cell = c("MS4A1", "CD79A", "CD79B", "CD19"),
  Myeloid = c("LYZ", "S100A8", "S100A9", "FCGR3A"),
  Macrophage = c("CD68", "C1QA", "C1QB", "CSF1R"),
  CAF = c("COL1A1", "COL1A2", "ACTA2", "FAP"),
  Endothelial = c("PECAM1", "VWF", "KDR", "ENG")
)

row_upper <- toupper(rownames(expr))
upper_to_real <- setNames(rownames(expr), row_upper)
gs <- lapply(immune_gene_sets, function(x) unique(upper_to_real[intersect(toupper(x), names(upper_to_real))]))
gs <- gs[lengths(gs) >= 2]

params <- gsvaParam(as.matrix(expr), gs, kcdf = "Gaussian", minSize = 2, maxSize = Inf)
ssgsea_scores <- gsva(params, verbose = FALSE)
write.csv(ssgsea_scores, file.path(OUTDIR, "ssGSEA_immune_scores.csv"))


## 6. CIBERSORT Skeleton

In [ ]:
if (RUN_CIBERSORT) {
  if (!file.exists(CIBERSORT_SCRIPT) || !file.exists(CIBERSORT_SIGNATURE)) {
    stop("CIBERSORT script/signature file not found. Check CIBERSORT_SCRIPT and CIBERSORT_SIGNATURE.")
  }
  source(CIBERSORT_SCRIPT)
  cib_input <- if (EXPR_IS_LOG) 2^expr - 1 else expr
  write.table(data.frame(GeneSymbol = rownames(cib_input), cib_input, check.names = FALSE),
              file.path(OUTDIR, "CIBERSORT_input.txt"), sep = "\t", quote = FALSE, row.names = FALSE)
  cib_res <- CIBERSORT(CIBERSORT_SIGNATURE, file.path(OUTDIR, "CIBERSORT_input.txt"), perm = 100, QN = TRUE)
  write.csv(cib_res, file.path(OUTDIR, "CIBERSORT_results.csv"))
}


## 7. Group Comparison and Heatmap

In [ ]:
score_df <- as.data.frame(t(ssgsea_scores)) %>% rownames_to_column(SAMPLE_COLUMN) %>% left_join(meta, by = SAMPLE_COLUMN)
score_long <- score_df %>% pivot_longer(cols = names(gs), names_to = "signature", values_to = "score")

p_box <- ggplot(score_long, aes(x = .data[[GROUP_COLUMN]], y = score, fill = .data[[GROUP_COLUMN]])) +
  geom_boxplot(outlier.shape = NA) + geom_jitter(width = 0.15, size = 1.5) +
  facet_wrap(~ signature, scales = "free_y") + labs(x = NULL, y = "ssGSEA score")
ggsave(file.path(OUTDIR, "ssGSEA_group_boxplot.pdf"), p_box, width = 12, height = 8)

ann <- data.frame(Group = meta[[GROUP_COLUMN]])
rownames(ann) <- meta[[SAMPLE_COLUMN]]
pheatmap(ssgsea_scores, annotation_col = ann, scale = "row", filename = file.path(OUTDIR, "ssGSEA_heatmap.pdf"), width = 8, height = 7)
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
